# CAP4770 Final Project – Feature Engineering & Preparation (Train/Val/Test Split, SMOTE)

## Objectives
- Split the numeric feature set into training, validation, and test sets to prevent data leakage.
- Build richer row-level features with log transforms, ratios, interactions, and cyclical encodings.
- Fit the scaler on the training split only, then transform validation and test splits, preferring cuML on GPU when available.
- Apply SMOTE to the training set to address class imbalance.
- Fit PCA on the engineered training set and transform validation and test sets for comparison.


In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the working directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())

In [1]:
import pandas as pd
import numpy as np

# Load numeric features before splitting
X = pd.read_csv("../data/X_numeric.csv")
y = pd.read_csv("../data/y_binary.csv").squeeze()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

X shape: (2916697, 8)
y shape: (2916697,)

Class distribution:
is_ransomware
0    2875284
1      41413
Name: count, dtype: int64


### Train/Validation/Test Split

This section creates the train, validation, and test partitions before any scaling, resampling, or dimensionality reduction so that later steps do not leak information across splits.


In [2]:
from sklearn.model_selection import train_test_split

# split train from rest of data
X_train, X_test_and_val, y_train, y_test_and_val = train_test_split(
    X, y, test_size=0.3, random_state=12, stratify=y
)

# split validation from test
X_val, X_test, y_val, y_test = train_test_split(
    X_test_and_val, y_test_and_val, test_size=0.5, random_state=12, stratify=y_test_and_val
)

print("Train size:", X_train.shape)
print("Val size:  ", X_val.shape)
print("Test size: ", X_test.shape)

Train size: (2041687, 8)
Val size:   (437505, 8)
Test size:  (437505, 8)


### Feature Engineering

This section builds richer predictors from the original numeric columns using log transforms, ratios, interactions, and cyclical encodings. These engineered features give the models more ways to capture transaction behavior patterns.


In [ ]:
base_feature_count = X_train.shape[1]


def engineer_features(df):
    df = df.copy()
    eps = 1.0

    nonnegative_cols = [col for col in ["length", "weight", "count", "looped", "neighbors", "income"] if col in df.columns]
    for col in nonnegative_cols:
        df[f"{col}_log1p"] = np.log1p(df[col].clip(lower=0))

    if {"income", "count"}.issubset(df.columns):
        df["income_per_count"] = df["income"] / (df["count"] + eps)
    if {"income", "neighbors"}.issubset(df.columns):
        df["income_per_neighbor"] = df["income"] / (df["neighbors"] + eps)
    if {"weight", "count"}.issubset(df.columns):
        df["weight_per_count"] = df["weight"] / (df["count"] + eps)
    if {"weight", "neighbors"}.issubset(df.columns):
        df["weight_per_neighbor"] = df["weight"] / (df["neighbors"] + eps)
    if {"count", "neighbors"}.issubset(df.columns):
        df["count_per_neighbor"] = df["count"] / (df["neighbors"] + eps)
        df["activity_pressure"] = np.log1p((df["count"].clip(lower=0) + 1) * (df["neighbors"].clip(lower=0) + 1))
    if {"looped", "count"}.issubset(df.columns):
        df["loop_ratio"] = df["looped"] / (df["count"] + eps)
    if {"length", "count"}.issubset(df.columns):
        df["length_per_count"] = df["length"] / (df["count"] + eps)
    if {"income", "weight"}.issubset(df.columns):
        df["income_weight_interaction"] = np.log1p((df["income"].clip(lower=0) + 1) * (df["weight"].clip(lower=0) + 1))
    if {"year", "day"}.issubset(df.columns):
        df["year_day_interaction"] = df["year"] * df["day"]
    if "day" in df.columns:
        df["day_sin"] = np.sin(2 * np.pi * df["day"] / 365.0)
        df["day_cos"] = np.cos(2 * np.pi * df["day"] / 365.0)

    return df


X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
feature_columns = X_train.columns.tolist()

print("Original feature count:", base_feature_count)
print("Engineered feature count:", len(feature_columns))
print("Added feature count:", len(feature_columns) - base_feature_count)


### Scale Using Training Data Only

This section standardizes the engineered features with statistics learned only from the training split. The same transformation is then applied to validation and test data so the feature scales stay consistent.


In [ ]:
GPU_ENABLED = False

try:
    import cudf
    from cuml.preprocessing import StandardScaler as cuStandardScaler
    GPU_ENABLED = True
    print("Using cuML StandardScaler on GPU.")
except ImportError:
    cudf = None
    from sklearn.preprocessing import StandardScaler as SkStandardScaler
    print("cuML not available; falling back to sklearn StandardScaler on CPU.")

if GPU_ENABLED:
    # cuML is taking over scaling here so this preprocessing step runs on the GPU when RAPIDS is installed in a Linux/NVIDIA CUDA environment; on macOS this notebook will fall back to CPU instead.
    scaler = cuStandardScaler()
    X_train = scaler.fit_transform(cudf.from_pandas(X_train.reset_index(drop=True))).to_pandas()
    X_val = scaler.transform(cudf.from_pandas(X_val.reset_index(drop=True))).to_pandas()
    X_test = scaler.transform(cudf.from_pandas(X_test.reset_index(drop=True))).to_pandas()
else:
    scaler = SkStandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_columns)
    X_val = pd.DataFrame(scaler.transform(X_val), columns=feature_columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_columns)

X_train = pd.DataFrame(X_train, columns=feature_columns)
X_val = pd.DataFrame(X_val, columns=feature_columns)
X_test = pd.DataFrame(X_test, columns=feature_columns)

print("Scaling complete using training-set statistics only.")


Scaling complete using training-set statistics only.


### SMOTE on Training Set

This section applies SMOTE only to the training split so the models see a less imbalanced class distribution during fitting while the validation and test splits remain untouched for honest evaluation.


In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=12)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
X_train_smote = pd.DataFrame(X_train_smote, columns=feature_columns)
y_train_smote = pd.Series(y_train_smote, name=y_train.name if getattr(y_train, "name", None) else "target")

print("Resampled train size:", X_train_smote.shape)
print("Resampled class distribution:")
print(pd.Series(y_train_smote).value_counts())


/Users/Ife/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Resampled train size: (4025396, 8)
Resampled class distribution:
is_ransomware
0    2012698
1    2012698
Name: count, dtype: int64


### PCA on Training Set

This section creates a lower-dimensional representation of the engineered features. The PCA version is saved alongside the full-feature version so later notebooks can compare whether dimensionality reduction helps or hurts performance.


In [5]:
if GPU_ENABLED:
    from cuml.decomposition import PCA as cuPCA

    X_train_gpu = cudf.from_pandas(X_train_smote.reset_index(drop=True))
    X_val_gpu = cudf.from_pandas(X_val.reset_index(drop=True))
    X_test_gpu = cudf.from_pandas(X_test.reset_index(drop=True))

    # cuML PCA expects an integer component count, so this pass finds the smallest count that explains at least 95% variance.
    pca_full = cuPCA(n_components=X_train_gpu.shape[1])
    pca_full.fit(X_train_gpu)

    explained = pca_full.explained_variance_ratio_.to_pandas().values
    cumulative = explained.cumsum()
    n_components_95 = int((cumulative >= 0.95).argmax() + 1)

    pca = cuPCA(n_components=n_components_95)
    X_train_pca = pca.fit_transform(X_train_gpu).to_pandas()
    X_val_pca = pca.transform(X_val_gpu).to_pandas()
    X_test_pca = pca.transform(X_test_gpu).to_pandas()
    variance_explained = float(cumulative[n_components_95 - 1])
else:
    from sklearn.decomposition import PCA as SkPCA

    pca = SkPCA(n_components=0.95, random_state=12)
    X_train_pca = pca.fit_transform(X_train_smote)
    X_val_pca = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)
    variance_explained = float(pca.explained_variance_ratio_.sum())

X_train_pca = pd.DataFrame(X_train_pca)
X_val_pca = pd.DataFrame(X_val_pca)
X_test_pca = pd.DataFrame(X_test_pca)

print(f"\nOriginal engineered features: {X_train_smote.shape[1]}")
print(f"PCA-reduced features:        {X_train_pca.shape[1]}")
print(f"Variance explained:          {variance_explained:.4f}")



Original features:    8
PCA-reduced features: 7
Variance explained:   0.9616


In [6]:
import os

for path in ["../data/X_numeric.csv", "../data/y_binary.csv"]:
    if os.path.exists(path):
        os.remove(path)

pd.DataFrame(X_train_smote, columns=feature_columns).to_csv("../data/X_train.csv", index=False)
pd.DataFrame(X_val, columns=feature_columns).to_csv("../data/X_val.csv", index=False)
pd.DataFrame(X_test, columns=feature_columns).to_csv("../data/X_test.csv", index=False)
pd.Series(y_train_smote).to_csv("../data/y_train.csv", index=False)
pd.Series(y_val).to_csv("../data/y_val.csv", index=False)
pd.Series(y_test).to_csv("../data/y_test.csv", index=False)

pd.DataFrame(X_train_pca).to_csv("../data/X_train_pca.csv", index=False)
pd.DataFrame(X_val_pca).to_csv("../data/X_val_pca.csv", index=False)
pd.DataFrame(X_test_pca).to_csv("../data/X_test_pca.csv", index=False)

print("All engineered splits saved to ../data/")


All splits saved to ../data/


## Next Steps

- Train baseline tree-based models on the engineered full-feature dataset.
- Compare the engineered full-feature results against PCA-reduced variants.
- Use the strongest full-feature models as the main candidates for widened hyperparameter search.
- Tune decision thresholds after model selection to improve final F1-score without retraining.
